In [1]:
import sys
sys.path.append('../')

import contextlib
import io
import time

import pandas as pd

from utilities import read_instance, read_solution, is_feasible, calculate_cost, score, size_class
from clarke_wright_solution import run_cw_ls_algo, time_budget
from route_first_cluster_second_solution import run_rfcs_algo
from yy_clarke_wright_solution import run_yy_cw_algo
from run_benchmarks import find_instances

In [2]:
# Each solver takes (demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit) -> (cost, routes)
SOLVERS = {
    'cw_ls': lambda demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit: run_cw_ls_algo(demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit=time_limit),
    # 'rfcs': lambda demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit: run_rfcs_algo(demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit=time_limit),
    'yy_cw': lambda demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit: run_yy_cw_algo(demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit=time_limit),

}

In [3]:
def run_benchmarks_df(solvers=SOLVERS, sets=('A', 'B', 'P', 'X'), samples_limit=None, time_vehicle_capacity=None):
    '''Runs every solver on the benchmark instances and returns one row per (solver, instance).'''
    rows = []
    for s in sets:

        # limit the number of samples if needed
        instances = list(find_instances([s]))
        if samples_limit:
            instances= instances[:samples_limit]

        for path, opt_path in instances:
            # read the test case and provided solution
            demand, dist_matrix, vehicle_capacity, num_vehicles = read_instance(path)
            optimal_cost = read_solution(opt_path)[0] if opt_path else None

            num_customers = len(demand) - 1
            for name, solver in solvers.items():
                time_start = time.perf_counter()
                _, routes = solver(demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit=None)
                elapsed = time.perf_counter() - time_start

                with contextlib.redirect_stdout(io.StringIO()):   # hide is_feasible's message
                    feasible = is_feasible(routes, demand, vehicle_capacity, num_vehicles)
                cost = calculate_cost(routes, dist_matrix)

                rows.append({
                    # instance variables
                    'solver': name, 'set': s, 'instance': path.stem,
                    'num_customers': num_customers, 'vehicle_capacity': vehicle_capacity, 'num_vehicles': num_vehicles,
                    'total_demandand': sum(demand),
                    # results
                    'time_taken': elapsed, 'cost': cost,
                    'optimal_cost': optimal_cost,
                    'gap_pct': 100 * (cost - optimal_cost) / optimal_cost if optimal_cost else None,
                    'feasible': feasible,
                })
                # print(f'{name:<6} {path.stem:<16} cost={cost:10.1f}  time={elapsed:6.1f}s', flush=True)
    return pd.DataFrame(rows)


# Quick run: first 3 instances of set A, vehicle_capacityped at 5 s each. Drop the limits for the full run.
results = run_benchmarks_df(sets=['A', 'B', 'P', 'X'], samples_limit=None, time_vehicle_capacity=5)


In [8]:
results.head()

,solver,set,instance,num_customers,vehicle_capacity,num_vehicles,total_demandand,time_taken,cost,optimal_cost,gap_pct,feasible
0,cw_ls,A,A-n32-k5,31,100,5,410,0.016381,827,784,5.484694,True
1,rfcs,A,A-n32-k5,31,100,5,410,0.430205,784,784,0.000000,True
2,yy_cw,A,A-n32-k5,31,100,5,410,24.003335,784,784,0.000000,True
3,cw_ls,A,A-n33-k5,32,100,5,446,0.006930,702,661,6.202723,True
4,rfcs,A,A-n33-k5,32,100,5,446,0.345413,661,661,0.000000,True


In [4]:
# compares the time taken and lowest cost
results.groupby(['set', 'solver'])[['cost', 'time_taken', 'gap_pct']].mean()

cost  time_taken   gap_pct
set solver                                    
A   cw_ls    1071.148148    0.024238  2.754736
    yy_cw    1043.925926   24.000328  0.168781
B   cw_ls     989.782609    0.025314  2.426921
    yy_cw     965.000000   24.000297  0.089028
P   cw_ls     618.041667    0.020539  5.569995
    yy_cw     588.208333   24.000151  0.111818
X   cw_ls   66471.920000    5.599878  6.111441
    yy_cw   64241.010000   78.481143  1.943118

In [5]:
results.groupby(['set', 'solver', 'feasible'])['instance'].count()

set  solver  feasible
A    cw_ls   True        27
     yy_cw   True        27
B    cw_ls   True        23
     yy_cw   True        23
P    cw_ls   True        24
     yy_cw   True        24
X    cw_ls   False        3
             True        97
     yy_cw   False        2
             True        98
Name: instance, dtype: int64

In [6]:
results


,solver,set,instance,num_customers,vehicle_capacity,num_vehicles,total_demandand,time_taken,cost,optimal_cost,gap_pct,feasible
0,cw_ls,A,A-n32-k5,31,100,5,410,0.011419,827,784,5.484694,True
1,yy_cw,A,A-n32-k5,31,100,5,410,24.001912,784,784,0.000000,True
2,cw_ls,A,A-n33-k5,32,100,5,446,0.012577,702,661,6.202723,True
3,yy_cw,A,A-n33-k5,32,100,5,446,24.000243,661,661,0.000000,True
4,cw_ls,A,A-n33-k6,32,100,6,541,0.011137,743,742,0.134771,True
...,...,...,...,...,...,...,...,...,...,...,...,...
343,yy_cw,X,X-n957-k87,956,11,87,956,144.000801,86155,85465,0.807348,True
344,cw_ls,X,X-n979-k58,978,998,58,57369,1.249581,123576,118976,3.866326,True
345,yy_cw,X,X-n979-k58,978,998,58,57369,144.004107,121223,118976,1.888616,True
346,cw_ls,X,X-n1001-k43,1000,131,43,5557,1.026372,76596,72355,5.861378,True
